In [2]:
import os
import sys
import math
import warnings
import contextlib
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

from astropy.table import Table, Column

from spisea import synthetic, atmospheres, reddening
from nbody6tools import Reader
from nbody62spisea import converter

sys.path.append('/home/wyz5rge/synthetic-cmd-dev/cmd_generator')
import interpolator

In [3]:
# --------------------------------------------------
# Input paths
# --------------------------------------------------

UPDATED_MERGED_ROOT = Path(
    '/home/wyz5rge/SPISEA/evolution/merged/'
    'baraffe_pisa_ekstrom_parsec/'
)

SIM_PATH = (
    '/standard/Tan_JC/backup_protoclusters/multiples/'
    'M3000new/sigma0p1/fiducial/sfe_ff003/00'
)

USE_ROTATING_MERGED = False


# --------------------------------------------------
# SPISEA settings
# --------------------------------------------------

AKs = 0.0
dist = 410
metallicity = 0.0

atm_func = atmospheres.get_BTSettl_2015_atmosphere
red_law = reddening.RedLawHosek18b()

# Generate all required filters simultaneously.
# This avoids SPISEA cache files that only contain a subset of filters.
ALL_FILTERS = [
    'jwst,F115W',
    'jwst,F162M',
    'jwst,F182M',
    'jwst,F200W',
]

FILTER_KEYS = {
    'F115W': 'm_jwst_F115W',
    'F162M': 'm_jwst_F162M',
    'F182M': 'm_jwst_F182M',
    'F200W': 'm_jwst_F200W',
}

ISO_CACHE_DIR = Path(
    '/standard/Tan_JC/backup_protoclusters/'
    'spisea_cache/merged_multifilter_comparison'
)

ISO_CACHE_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# Theoretical age grid
# --------------------------------------------------

AGE_MIN_MYR = 1.0
AGE_MAX_MYR = 20.0
AGE_STEP_MYR = 0.5

ISO_AGES_MYR = np.arange(
    AGE_MIN_MYR,
    AGE_MAX_MYR + 0.5 * AGE_STEP_MYR,
    AGE_STEP_MYR
)

LOG_AGES = np.log10(ISO_AGES_MYR * 1e6)

print(f'Number of isochrones: {len(ISO_AGES_MYR)}')
print('Age range:', ISO_AGES_MYR[0], 'to', ISO_AGES_MYR[-1], 'Myr')

Number of isochrones: 39
Age range: 1.0 to 20.0 Myr


In [4]:
class MergedBaraffePisaEkstromParsecDAT:
    """
    Reader for the merged Baraffe-Pisa-Ekstrom-PARSEC evolution model
    stored in ASCII iso_*.dat files.

    root_dir must contain:
        z015_rot/
        z015_norot/
    """

    def __init__(self, root_dir, rot=False):
        self.root_dir = Path(root_dir).expanduser().resolve()
        self.rot = bool(rot)

        self.model_dir = str(self.root_dir)
        self.z_list = [0.015]
        self.z_solar = 0.015
        self.mass_list = []

        self.grid_dir = self.root_dir / (
            'z015_rot' if self.rot else 'z015_norot'
        )

        if not self.grid_dir.is_dir():
            raise FileNotFoundError(
                f'Merged-model directory not found: {self.grid_dir}'
            )

        paths = sorted(self.grid_dir.glob('iso_*.dat'))

        if len(paths) == 0:
            raise FileNotFoundError(
                f'No iso_*.dat files found in {self.grid_dir}'
            )

        self.age_file_map = {}

        for path in paths:
            try:
                log_age = float(path.stem.split('_')[1])
            except (IndexError, ValueError):
                continue

            self.age_file_map[round(log_age, 2)] = path

        self.age_list = np.array(
            sorted(self.age_file_map.keys()),
            dtype=float
        )

    def isochrone(self, age=1.0e6, metallicity=0.0):
        log_age_requested = math.log10(age)

        if log_age_requested < self.age_list[0]:
            raise ValueError(
                f'Requested logAge={log_age_requested:.4f} is younger '
                f'than the grid minimum logAge={self.age_list[0]:.2f}.'
            )

        if log_age_requested > self.age_list[-1]:
            raise ValueError(
                f'Requested logAge={log_age_requested:.4f} is older '
                f'than the grid maximum logAge={self.age_list[-1]:.2f}.'
            )

        # Select the nearest available native merged-model age.
        idx = np.argmin(
            np.abs(self.age_list - log_age_requested)
        )
        selected_log_age = float(self.age_list[idx])

        iso_path = self.age_file_map[
            round(selected_log_age, 2)
        ]

        dtype = [
            ('mass', 'f8'),
            ('logT', 'f8'),
            ('logL', 'f8'),
            ('logg', 'f8'),
            ('logT_WR', 'f8'),
            ('mass_current', 'f8'),
            ('phase', 'i4'),
            ('model_ref', 'U32'),
        ]

        data = np.genfromtxt(
            str(iso_path),
            comments='#',
            dtype=dtype,
            encoding='utf-8'
        )

        data = np.atleast_1d(data)
        iso = Table(data)

        is_wr = ~np.isclose(
            np.asarray(iso['logT'], dtype=float),
            np.asarray(iso['logT_WR'], dtype=float),
            rtol=0.0,
            atol=1e-8
        )

        iso.add_column(
            Column(is_wr, name='isWR')
        )

        iso.meta['log_age'] = selected_log_age
        iso.meta['log_age_requested'] = log_age_requested
        iso.meta['metallicity_in'] = metallicity
        iso.meta['metallicity_act'] = 0.0
        iso.meta['source_file'] = str(iso_path)

        return iso

In [5]:
evo_model = MergedBaraffePisaEkstromParsecDAT(
    UPDATED_MERGED_ROOT,
    rot=USE_ROTATING_MERGED
)

print('Model class:', type(evo_model))
print('Grid directory:', evo_model.grid_dir)
print(
    'Available log-age range:',
    evo_model.age_list.min(),
    'to',
    evo_model.age_list.max()
)

test_iso = evo_model.isochrone(
    age=1.0e6,
    metallicity=metallicity
)

print('Test source file:', test_iso.meta['source_file'])
print(
    'Physical mass range:',
    np.min(test_iso['mass']),
    'to',
    np.max(test_iso['mass']),
    'Msun'
)

Model class: <class '__main__.MergedBaraffePisaEkstromParsecDAT'>
Grid directory: /sfs/gpfs/tardis/home/wyz5rge/SPISEA/evolution/merged/baraffe_pisa_ekstrom_parsec/z015_norot
Available log-age range: 6.0 to 10.09
Test source file: /sfs/gpfs/tardis/home/wyz5rge/SPISEA/evolution/merged/baraffe_pisa_ekstrom_parsec/z015_norot/iso_6.00.dat
Physical mass range: 0.01 to 500.0 Msun


In [6]:
def build_multifilter_isochrone_grid():
    iso_grid = []
    records = []

    for age_myr, log_age in zip(ISO_AGES_MYR, LOG_AGES):
        print(f'Building age = {age_myr:.1f} Myr')

        try:
            iso = synthetic.IsochronePhot(
                log_age,
                AKs,
                dist,
                metallicity=metallicity,
                evo_model=evo_model,
                atm_func=atm_func,
                red_law=red_law,
                filters=ALL_FILTERS,
                iso_dir=str(ISO_CACHE_DIR)
            )

            missing = [
                key for key in FILTER_KEYS.values()
                if key not in iso.points.colnames
            ]

            if missing:
                raise KeyError(
                    f'Isochrone is missing filter columns: {missing}'
                )

            iso_grid.append(iso)

            mass = np.asarray(
                iso.points['mass'],
                dtype=float
            )

            records.append({
                'age_myr': age_myr,
                'log_age': log_age,
                'n_points': len(iso.points),
                'mass_min': np.nanmin(mass),
                'mass_max': np.nanmax(mass),
                'status': 'success',
                'error': '',
            })

        except Exception as exc:
            print(f'FAILED at {age_myr:.1f} Myr: {exc}')

            iso_grid.append(None)

            records.append({
                'age_myr': age_myr,
                'log_age': log_age,
                'n_points': 0,
                'mass_min': np.nan,
                'mass_max': np.nan,
                'status': 'failed',
                'error': str(exc),
            })

    return iso_grid, pd.DataFrame(records)


ISO_GRID, df_iso_coverage = build_multifilter_isochrone_grid()

display(df_iso_coverage)

Building age = 1.0 Myr
Changing to T=  7000 for T=  7029 logg=3.33
Changing to T=  7000 for T=  7066 logg=3.33
Changing to T=  7000 for T=  7104 logg=3.34
Changing to T=  7000 for T=  7142 logg=3.34
Changing to T=  7000 for T=  7180 logg=3.34
Changing to T=  7000 for T=  7219 logg=3.35
Changing to T=  7000 for T=  7258 logg=3.35
Changing to T=  7000 for T=  7298 logg=3.35
Changing to T=  7000 for T=  7338 logg=3.36
Changing to T=  7000 for T=  7379 logg=3.36
Changing to T=  7000 for T=  7420 logg=3.36
Changing to T=  7000 for T=  7461 logg=3.37
Changing to T=  7000 for T=  7502 logg=3.37
Changing to T=  7000 for T=  7544 logg=3.37
Changing to T=  7000 for T=  7588 logg=3.38
Changing to T=  7000 for T=  7630 logg=3.38
Changing to T=  7000 for T=  7674 logg=3.38
Changing to T=  7000 for T=  7718 logg=3.39
Changing to T=  7000 for T=  7762 logg=3.39
Changing to T=  7000 for T=  7807 logg=3.39
Changing to T=  7000 for T=  7854 logg=3.40
Changing to T=  7000 for T=  7900 logg=3.40
Changing 

KeyboardInterrupt: 

In [7]:
DIAGRAM_CHOICES = [
    {
        'kind': 'hr',
        'title': r'Luminosity vs. $T_{\rm eff}$',
        'xlabel': r'$T_{\rm eff}$ (K)',
        'ylabel': r'$\log_{10}(L/L_\odot)$',
    },

    {
        'kind': 'cmd',
        'color_a': 'F115W',
        'color_b': 'F200W',
        'y_filter': 'F200W',
        'title': 'F115W-F200W vs F200W',
    },

    {
        'kind': 'cmd',
        'color_a': 'F162M',
        'color_b': 'F182M',
        'y_filter': 'F162M',
        'title': 'F162M-F182M vs F162M',
    },

    {
        'kind': 'cmd',
        'color_a': 'F162M',
        'color_b': 'F182M',
        'y_filter': 'F182M',
        'title': 'F162M-F182M vs F182M',
    },

    {
        'kind': 'cmd',
        'color_a': 'F162M',
        'color_b': 'F200W',
        'y_filter': 'F200W',
        'title': 'F162M-F200W vs F200W',
    },

    {
        'kind': 'cmd',
        'color_a': 'F182M',
        'color_b': 'F200W',
        'y_filter': 'F200W',
        'title': 'F182M-F200W vs F200W',
    },
]

In [8]:
L_SUN_WATTS = 3.846e26


def finite_iso_points(iso):
    if iso is None:
        return None

    pts = iso.points

    required = [
        'L',
        'Teff',
        'mass',
        *FILTER_KEYS.values(),
    ]

    missing = [
        col for col in required
        if col not in pts.colnames
    ]

    if missing:
        raise KeyError(
            f'Missing isochrone columns: {missing}'
        )

    return pts


def get_iso_diagram_coordinates(iso, diagram):
    pts = finite_iso_points(iso)

    if pts is None:
        return np.array([]), np.array([]), np.array([])

    mass = np.asarray(pts['mass'], dtype=float)

    if diagram['kind'] == 'hr':
        teff = np.asarray(pts['Teff'], dtype=float)
        luminosity_watts = np.asarray(pts['L'], dtype=float)

        x = teff
        y = np.log10(luminosity_watts / L_SUN_WATTS)

    else:
        key_a = FILTER_KEYS[diagram['color_a']]
        key_b = FILTER_KEYS[diagram['color_b']]
        key_y = FILTER_KEYS[diagram['y_filter']]

        mag_a = np.asarray(pts[key_a], dtype=float)
        mag_b = np.asarray(pts[key_b], dtype=float)
        ymag = np.asarray(pts[key_y], dtype=float)

        x = mag_a - mag_b
        y = ymag

    good = (
        np.isfinite(x) &
        np.isfinite(y) &
        np.isfinite(mass)
    )

    return x[good], y[good], mass[good]

In [9]:
def plot_theoretical_multidiagram_grid(
    iso_grid,
    ages_myr,
    diagram_choices
):
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(16, 10)
    )
    axes = axes.flatten()

    cmap = plt.get_cmap('coolwarm')
    norm = Normalize(
        vmin=np.min(ages_myr),
        vmax=np.max(ages_myr)
    )

    for ax, diagram in zip(axes, diagram_choices):
        all_min_masses = []
        all_max_masses = []

        for iso, age_myr in zip(iso_grid, ages_myr):
            if iso is None:
                continue

            x, y, mass = get_iso_diagram_coordinates(
                iso,
                diagram
            )

            if len(x) == 0:
                continue

            ax.plot(
                x,
                y,
                color=cmap(norm(age_myr)),
                lw=1.0,
                alpha=0.68
            )

            all_min_masses.append(np.min(mass))
            all_max_masses.append(np.max(mass))

        if diagram['kind'] == 'hr':
            ax.set_xlabel(diagram['xlabel'])
            ax.set_ylabel(diagram['ylabel'])
            ax.invert_xaxis()
        else:
            ax.set_xlabel(
                f"{diagram['color_a']} - "
                f"{diagram['color_b']}"
            )
            ax.set_ylabel(diagram['y_filter'])
            ax.invert_yaxis()

        if len(all_min_masses) > 0:
            coverage_text = (
                'Photometric mass range\n'
                f'{min(all_min_masses):.3f}-'
                f'{max(all_max_masses):.1f} '
                r'$M_\odot$'
            )

            ax.text(
                0.03,
                0.97,
                coverage_text,
                transform=ax.transAxes,
                ha='left',
                va='top',
                fontsize=9,
                bbox=dict(
                    boxstyle='round',
                    facecolor='white',
                    alpha=0.82
                )
            )

        ax.set_title(diagram['title'])
        ax.grid(alpha=0.22)

    sm = ScalarMappable(
        norm=norm,
        cmap=cmap
    )
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=axes.tolist(),
        pad=0.015,
        fraction=0.025
    )
    cbar.set_label('Isochrone age (Myr)')

    rotation_text = (
        'rotating high-mass Ekstrom component'
        if USE_ROTATING_MERGED
        else 'nonrotating high-mass Ekstrom component'
    )

    fig.suptitle(
        'Merged-model isochrone geometry across diagram spaces\n'
        f'1-20 Myr; {rotation_text}',
        fontsize=16
    )

    plt.subplots_adjust(
        top=0.90,
        right=0.90,
        wspace=0.27,
        hspace=0.30
    )

    plt.show()


plot_theoretical_multidiagram_grid(
    ISO_GRID,
    ISO_AGES_MYR,
    DIAGRAM_CHOICES
)

NameError: name 'ISO_GRID' is not defined

In [10]:
SNAPSHOT_TIME_MYR = 1.5


def load_cluster_table(sim_path, snapshot_time_myr):
    sim_path = os.path.abspath(sim_path)

    if not sim_path.endswith('/'):
        sim_path += '/'

    print('Reading snapshot from:')
    print(sim_path)

    snapshot = Reader.read_snapshot(
        sim_path,
        time=snapshot_time_myr
    )

    snapshot.to_physical()

    return converter.to_spicea_table(snapshot)


cluster_table = load_cluster_table(
    SIM_PATH,
    SNAPSHOT_TIME_MYR
)

print(cluster_table)
print('Number of simulated systems:', len(cluster_table))

Reading snapshot from:
/standard/Tan_JC/backup_protoclusters/multiples/M3000new/sigma0p1/fiducial/sfe_ff003/00/
        mass         isMultiple ...         age        
-------------------- ---------- ... -------------------
 0.19400843173671706        0.0 ...  0.9703966000072413
 0.07426646486409774        0.0 ...  1.2324075798870435
 0.04664335740934216        0.0 ...  1.2322126988328113
                 ...        ... ...                 ...
 0.15460383712183084        1.0 ...  0.1622791708991409
   0.296217352154055        1.0 ... 0.28446959190287013
  0.8898785064344591        1.0 ...  0.2921592735011245
0.057220117036550555        1.0 ... 0.12035132408542792
Length = 756 rows
Number of simulated systems: 756


In [11]:
def safe_interpolate(
    age_myr,
    mass,
    iso_grid,
    log_age_arr,
    filter_pair
):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')

            with contextlib.redirect_stdout(io.StringIO()):
                with contextlib.redirect_stderr(io.StringIO()):
                    result = interpolator.interpolate(
                        age_myr,
                        mass,
                        iso_grid,
                        log_age_arr,
                        filter_pair
                    )

        if result is None:
            return None

        result = np.asarray(result, dtype=float)

        if not np.all(np.isfinite(result)):
            return None

        return result

    except Exception:
        return None

In [12]:
def build_common_snapshot_catalog(
    cluster_table,
    iso_grid,
    log_age_arr
):
    masses = np.asarray(
        cluster_table['mass'],
        dtype=float
    )
    ages_myr = np.asarray(
        cluster_table['age'],
        dtype=float
    )

    cmd_diagrams = [
        d for d in DIAGRAM_CHOICES
        if d['kind'] == 'cmd'
    ]

    rows = []
    rejection_counts = {
        'age_below_grid': 0,
        'age_above_grid': 0,
        'interpolation_failure': 0,
        'retained': 0,
    }

    grid_ages_myr = (
        np.power(10.0, np.asarray(log_age_arr)) / 1e6
    )

    for system_index, (mass, age_myr) in enumerate(
        zip(masses, ages_myr)
    ):
        if age_myr < grid_ages_myr[0]:
            rejection_counts['age_below_grid'] += 1
            continue

        if age_myr > grid_ages_myr[-1]:
            rejection_counts['age_above_grid'] += 1
            continue

        row = {
            'system_index': system_index,
            'mass': mass,
            'age_myr': age_myr,
        }

        successful = True
        reference_luminosity = None
        reference_teff = None
        reference_logg = None

        for diagram in cmd_diagrams:
            key_a = FILTER_KEYS[diagram['color_a']]
            key_b = FILTER_KEYS[diagram['color_b']]

            star = safe_interpolate(
                age_myr,
                mass,
                iso_grid,
                log_age_arr,
                [key_a, key_b]
            )

            if star is None:
                successful = False
                break

            # interpolator output:
            # [luminosity, Teff, logg, magnitude_a, magnitude_b]
            luminosity = float(star[0])
            teff = float(star[1])
            logg = float(star[2])
            mag_a = float(star[3])
            mag_b = float(star[4])

            if reference_luminosity is None:
                reference_luminosity = luminosity
                reference_teff = teff
                reference_logg = logg

            row[
                f"color_{diagram['color_a']}_{diagram['color_b']}"
            ] = mag_a - mag_b

            if diagram['y_filter'] == diagram['color_a']:
                ymag = mag_a
            elif diagram['y_filter'] == diagram['color_b']:
                ymag = mag_b
            else:
                raise ValueError(
                    'The y filter must be one of the two '
                    'filters passed to the interpolator.'
                )

            row[
                f"ymag_{diagram['title']}"
            ] = ymag

        if not successful:
            rejection_counts['interpolation_failure'] += 1
            continue

        row['luminosity_watts'] = reference_luminosity
        row['log_luminosity_lsun'] = np.log10(
            reference_luminosity / L_SUN_WATTS
        )
        row['teff'] = reference_teff
        row['logg'] = reference_logg

        rows.append(row)
        rejection_counts['retained'] += 1

    return pd.DataFrame(rows), rejection_counts


df_snapshot, snapshot_rejections = build_common_snapshot_catalog(
    cluster_table,
    ISO_GRID,
    LOG_AGES
)

print('Snapshot interpolation results:')
for reason, count in snapshot_rejections.items():
    print(f'  {reason}: {count}')

print('\nCommon retained sample:', len(df_snapshot))
display(df_snapshot.head())

NameError: name 'ISO_GRID' is not defined

In [13]:
def plot_snapshot_multidiagram_comparison(
    df_snapshot,
    iso_grid,
    ages_myr,
    snapshot_time_myr,
    diagram_choices
):
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(16, 10)
    )
    axes = axes.flatten()

    cmap = plt.get_cmap('coolwarm')
    norm = Normalize(
        vmin=np.min(ages_myr),
        vmax=np.max(ages_myr)
    )

    for ax, diagram in zip(axes, diagram_choices):
        # Plot theoretical isochrone grid lightly.
        for iso, age_myr in zip(iso_grid, ages_myr):
            if iso is None:
                continue

            x_iso, y_iso, _ = get_iso_diagram_coordinates(
                iso,
                diagram
            )

            if len(x_iso) == 0:
                continue

            ax.plot(
                x_iso,
                y_iso,
                color=cmap(norm(age_myr)),
                lw=0.8,
                alpha=0.25,
                zorder=1
            )

        # Plot the common simulation-star sample.
        if diagram['kind'] == 'hr':
            x_star = df_snapshot['teff']
            y_star = df_snapshot['log_luminosity_lsun']

            ax.set_xlabel(r'$T_{\rm eff}$ (K)')
            ax.set_ylabel(r'$\log_{10}(L/L_\odot)$')
            ax.invert_xaxis()

        else:
            color_col = (
                f"color_{diagram['color_a']}_"
                f"{diagram['color_b']}"
            )
            y_col = f"ymag_{diagram['title']}"

            x_star = df_snapshot[color_col]
            y_star = df_snapshot[y_col]

            ax.set_xlabel(
                f"{diagram['color_a']} - "
                f"{diagram['color_b']}"
            )
            ax.set_ylabel(diagram['y_filter'])
            ax.invert_yaxis()

        ax.scatter(
            x_star,
            y_star,
            s=6,
            color='black',
            alpha=0.50,
            edgecolors='none',
            zorder=3
        )

        ax.text(
            0.03,
            0.97,
            f'N = {len(df_snapshot)}',
            transform=ax.transAxes,
            ha='left',
            va='top',
            fontsize=9,
            bbox=dict(
                boxstyle='round',
                facecolor='white',
                alpha=0.82
            )
        )

        ax.set_title(diagram['title'])
        ax.grid(alpha=0.22)

    sm = ScalarMappable(
        norm=norm,
        cmap=cmap
    )
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=axes.tolist(),
        pad=0.015,
        fraction=0.025
    )
    cbar.set_label('Isochrone age (Myr)')

    fig.suptitle(
        'Same simulated cluster snapshot across diagram spaces\n'
        rf'$\Sigma_{{\rm cloud}}=0.1$ g cm$^{{-2}}$, '
        rf'$\epsilon_{{\rm ff}}=0.03$, '
        f't = {snapshot_time_myr:.2f} Myr; primary stars',
        fontsize=16
    )

    plt.subplots_adjust(
        top=0.89,
        right=0.90,
        wspace=0.27,
        hspace=0.30
    )

    plt.show()


plot_snapshot_multidiagram_comparison(
    df_snapshot,
    ISO_GRID,
    ISO_AGES_MYR,
    SNAPSHOT_TIME_MYR,
    DIAGRAM_CHOICES
)

NameError: name 'df_snapshot' is not defined

In [ ]:
range_rows = []

for diagram in DIAGRAM_CHOICES:
    if diagram['kind'] == 'hr':
        x = df_snapshot['teff'].values
        y = df_snapshot['log_luminosity_lsun'].values
    else:
        x = df_snapshot[
            f"color_{diagram['color_a']}_{diagram['color_b']}"
        ].values
        y = df_snapshot[
            f"ymag_{diagram['title']}"
        ].values

    range_rows.append({
        'diagram': diagram['title'],
        'x_min': np.nanmin(x),
        'x_max': np.nanmax(x),
        'y_min': np.nanmin(y),
        'y_max': np.nanmax(y),
    })

df_panel_ranges = pd.DataFrame(range_rows)
display(df_panel_ranges)

In [ ]:
'''
if save_path is not None:
    fig.savefig(
        save_path,
        dpi=300,
        bbox_inches='tight'
    )

OUTPUT_DIR = Path('science-images')
OUTPUT_DIR.mkdir(exist_ok=True)

THEORETICAL_FIGURE = (
    OUTPUT_DIR /
    'fig_merged_isochrone_grid_multifilter.png'
)

SNAPSHOT_FIGURE = (
    OUTPUT_DIR /
    'fig_merged_snapshot_multifilter.png'
)
'''